In [ ]:
from __future__ import annotations
import gzip
from pathlib import Path
import pandas as pd
import requests # untuk mengakses download melalui internet
from tqdm.auto import tqdm

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Info: Tidak berjalan di Google Colab. Melanjutkan menggunakan path lokal.')

if IN_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/skripsi/data')
else:
    BASE_DIR = Path('./data')

CLINVAR_DIR = BASE_DIR / 'ClinVar'
WGS_DIR = BASE_DIR / 'WGS_WestJava'
DBNSFP_DIR = BASE_DIR / 'dbNSFP'

PREVIEW_CLINVAR_CSV = BASE_DIR / 'Lampiran_Raw_ClinVar.csv'
PREVIEW_WGS_CSV = BASE_DIR / 'Lampiran_Raw_WGS.csv'
PREVIEW_DBNSFP_CSV = BASE_DIR / 'Lampiran_Raw_dbNSFP.csv'
DOWNLOAD_STATUS_CSV = BASE_DIR / 'download_status.csv'

clinvar_file = CLINVAR_DIR / 'variant_summary.txt.gz'
wgs_file     = WGS_DIR / 'WGS_WestJava_227.vcf.gz'
dbnsfp_file  = DBNSFP_DIR / 'dbNSFP5.3a_grch38.gz'
dbnsfp_index = DBNSFP_DIR / 'dbNSFP5.3a_grch38.gz.tbi'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. PERSIAPAN FOLDER


In [ ]:
for folder in [BASE_DIR, CLINVAR_DIR, WGS_DIR, DBNSFP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('\nStruktur folder yang digunakan:')
print(f'   ClinVar      : {CLINVAR_DIR}')
print(f'   WGS WestJava : {WGS_DIR}')
print(f'   dbNSFP       : {DBNSFP_DIR}')
print('Semua folder siap!\n')



Struktur folder yang digunakan:
   ClinVar      : /content/drive/MyDrive/skripsi/data/ClinVar
   WGS WestJava : /content/drive/MyDrive/skripsi/data/WGS_WestJava
   dbNSFP       : /content/drive/MyDrive/skripsi/data/dbNSFP
Semua folder siap!



## 2. FUNGSI BANTU


In [ ]:
# Mengecek file gzip apakah valid atau corrupt
def is_valid_gzip(file_path: Path) -> bool:
    try:
        with gzip.open(file_path, 'rb') as handle:
            handle.read(1)
        return True
    except OSError:
        return False

# Memastikan file telah ada di direktori
def file_ready(output_path: Path) -> bool:
    if not output_path.exists():
        return False
    if output_path.suffix == '.gz':
        return is_valid_gzip(output_path)
    return output_path.stat().st_size > 0

# Mengubah ukuran byte menjadi format yang mudah dibaca
def human_size(num_bytes: int) -> str:
    if num_bytes >= 1e9:
        return f'{num_bytes / 1e9:.2f} GB'
    if num_bytes >= 1e6:
        return f'{num_bytes / 1e6:.1f} MB'
    if num_bytes >= 1e3:
        return f'{num_bytes / 1e3:.1f} KB'
    return f'{num_bytes} B'

# Mendownload file besar dengan auto resume
def download_file(url: str, output_path: Path, description: str = '') -> int:
    if file_ready(output_path):
        print(f'\n{output_path.name} sudah ada, melewati download.')
        print(f'   Lokasi : {output_path}')
        print(f'   Ukuran : {human_size(output_path.stat().st_size)}')
        return 0

    print(f'\n{description}')
    print(f'   URL    : {url}')
    print(f'   Output : {output_path}')

    import time

    max_retries = 10
    retry_count = 0

    while retry_count < max_retries:
        initial_pos = 0
        mode = 'wb'
        headers = {'Accept-Encoding': 'identity'}

        if output_path.exists():
            initial_pos = output_path.stat().st_size
            if initial_pos > 0:
                print(f'   Melanjutkan download dari {human_size(initial_pos)}...')

                headers['Range'] = f'bytes={initial_pos}-'
                mode = 'ab'

        try:
            response = requests.get(url, headers=headers, stream=True, timeout=(15, 60))

            if response.status_code == 416:
                print('   Selesai! File sudah terdownload penuh.')
                return 0
            response.raise_for_status()

            if initial_pos > 0 and response.status_code != 206:
                print('   Server mengabaikan resume. Mengulang dari 0...')
                initial_pos = 0
                mode = 'wb'
                try:
                    output_path.unlink()
                except FileNotFoundError:
                    pass

            total_size = int(response.headers.get('content-length', 0)) + initial_pos

            with open(output_path, mode) as handle, tqdm(
                desc=output_path.name,
                total=total_size,
                initial=initial_pos,
                unit='iB',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        written = handle.write(chunk)
                        bar.update(written)

            if output_path.suffix == '.gz' and not is_valid_gzip(output_path):
                retry_count += 1
                print('   File gzip tidak valid. Menghapus lalu mencoba ulang...')
                try:
                    output_path.unlink()
                except FileNotFoundError:
                    pass
                if retry_count < max_retries:
                    time.sleep(5)
                    continue
                print(f'   Gagal total setelah {max_retries} percobaan.')
                return 1

            print(f'   Selesai! Ukuran: {human_size(output_path.stat().st_size)}')
            return 0

        except requests.exceptions.RequestException:
            retry_count += 1
            print('\n   Koneksi terputus atau timeout.')
            if retry_count < max_retries:
                print(f'   Mencoba kembali ({retry_count}/{max_retries}) dalam 5 detik...')
                time.sleep(5)
            else:
                print(f'   Gagal total setelah {max_retries} percobaan.')
                return 1

# Mencetak tabel dataframe ke terminal dengan garis rapi
def print_grid_table(preview_df: pd.DataFrame) -> None:
    """Cetak DataFrame kecil dalam bentuk tabel box yang rapi tanpa memotong isi."""
    if preview_df.empty:
        print('   (tidak ada data untuk ditampilkan)')
        return

    display_df = preview_df.copy().fillna('')
    display_df = display_df.astype(str).applymap(lambda text: str(text).replace('\n', ' ').replace('\r', ' '))
    headers = [str(column).replace('\n', ' ').replace('\r', ' ') for column in display_df.columns]
    rows = display_df.values.tolist()

    widths = []
    for index, header in enumerate(headers):
        column_cells = [row[index] for row in rows]
        cell_width = max([len(header), *[len(cell) for cell in column_cells], 3])
        widths.append(cell_width)

    def format_row(values) -> str:
        formatted_cells = []
        for index, value in enumerate(values):
            formatted_cells.append(f' {str(value).ljust(widths[index])} ')
        return '│' + '│'.join(formatted_cells) + '│'

    def border(left: str, fill: str, middle: str, right: str) -> str:
        return left + middle.join(fill * (width + 2) for width in widths) + right

    top_border = border('┌', '─', '┬', '┐')
    separator = border('├', '─', '┼', '┤')
    bottom_border = border('└', '─', '┴', '┘')

    print(top_border)
    print(format_row(headers))
    print(separator)
    for row in rows:
        print(format_row(row))
    print(bottom_border)

# Membaca sebagian kecil isi file tanpa memuat semuanya ke RAM
def preview_table(file_path: Path, file_type: str, nrows: int = 10):
    print(f'\nPreview tabel {file_type} ({nrows} baris teratas):')
    try:
        if file_type == 'ClinVar':
            preview = pd.read_csv(file_path, sep='\t', compression='gzip', nrows=nrows, dtype=str)
            print(f'   Baris (sample): {nrows} dari total | Kolom: {preview.shape[1]}')
            print_grid_table(preview)
            return preview
        if file_type == 'dbNSFP':
            header = pd.read_csv(file_path, sep='\t', compression='gzip', nrows=0)
            preview = pd.read_csv(
                file_path,
                sep='\t',
                compression='gzip',
                nrows=nrows,
                usecols=header.columns[: min(8, len(header.columns))],
                dtype=str,
            )
            print(f'   Baris (sample): {nrows} dari total | Kolom ditampilkan: {preview.shape[1]} dari {len(header.columns)} total')
            print_grid_table(preview)
            return preview

        if file_type == 'WGS':
            with gzip.open(file_path, 'rt') as handle:
                rows = []
                for line in handle:
                    if line.startswith('#'):
                        continue
                    parts = line.rstrip('\n').split('\t')
                    if len(parts) >= 5:
                        rows.append([parts[0], parts[1], parts[3], parts[4]])
                    if len(rows) >= nrows:
                        break
            preview = pd.DataFrame(rows, columns=['CHROM', 'POS', 'REF', 'ALT'])
            print(f'   Baris (sample): {len(preview)} | Kolom: {preview.shape[1]} (CHROM, POS, REF, ALT)')
            print_grid_table(preview)
            return preview
    except Exception as exc:
        print(f'   Gagal membuat preview {file_type}: {exc}')
        return None
    return None

# Menyimpan hasilnya ke dalam format CSV
def save_preview_csv(preview_df: pd.DataFrame | None, preview_path: Path) -> None:
    if preview_df is None:
        return
    preview_df.to_csv(preview_path, index=False)
    print(f'   Preview CSV disimpan di: {preview_path}')

## 3. DOWNLOAD CLINVAR


In [ ]:
# National Center for Biotechnology Information
print('=' * 60)
print('DOWNLOAD CLINVAR - variant_summary.txt.gz')
print('=' * 60)
clinvar_file = CLINVAR_DIR / 'variant_summary.txt.gz'
download_file(
    url='https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz',
    output_path=clinvar_file,
    description='ClinVar Variant Summary (~439 MB)',
)
clinvar_preview = preview_table(clinvar_file, 'ClinVar', nrows=20)
save_preview_csv(clinvar_preview, PREVIEW_CLINVAR_CSV)

DOWNLOAD CLINVAR - variant_summary.txt.gz

variant_summary.txt.gz sudah ada, melewati download.
   Lokasi : /content/drive/MyDrive/skripsi/data/ClinVar/variant_summary.txt.gz
   Ukuran : 441.8 MB

Preview tabel ClinVar (20 baris teratas):
   Baris (sample): 20 dari total | Kolom: 43
┌───────────┬───────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────┬────────┬────────────┬────────────┬─────────────────────────────────────────────────────┬───────────────┬───────────────┬─────────────┬─────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

/tmp/ipykernel_2679/74636231.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display_df = display_df.astype(str).applymap(lambda text: str(text).replace('\n', ' ').replace('\r', ' '))


   Preview CSV disimpan di: /content/drive/MyDrive/skripsi/data/Lampiran_Raw_ClinVar.csv


## 4. DOWNLOAD WGS WEST JAVA

In [ ]:
# FTP European Nucleotide Archive (ENA)
print('\n' + '=' * 60)
print('DOWNLOAD WGS WEST JAVA (ENA: PRJEB85629)')
print('=' * 60)
wgs_file = WGS_DIR / 'WGS_WestJava_227.vcf.gz'
download_file(
    url='https://ftp.sra.ebi.ac.uk/vol1/analysis/ERZ250/ERZ25039609/WGS_WestJava_227.vcf.gz',
    output_path=wgs_file,
    description='WGS West Java VCF - 227 sampel Jawa Barat (~1.7 GB)',
)
wgs_preview = preview_table(wgs_file, 'WGS', nrows=20)
save_preview_csv(wgs_preview, PREVIEW_WGS_CSV)

DBNSFP_VERSION = '5.3a'
DBNSFP_BASE_URL = 'https://dist.genos.us/academic/b2fd38'


DOWNLOAD WGS WEST JAVA (ENA: PRJEB85629)

WGS_WestJava_227.vcf.gz sudah ada, melewati download.
   Lokasi : /content/drive/MyDrive/skripsi/data/WGS_WestJava/WGS_WestJava_227.vcf.gz
   Ukuran : 1.71 GB

Preview tabel WGS (20 baris teratas):
   Baris (sample): 20 | Kolom: 4 (CHROM, POS, REF, ALT)
┌───────┬───────┬─────┬─────┐
│ CHROM │ POS   │ REF │ ALT │
├───────┼───────┼─────┼─────┤
│ 1     │ 10177 │ A   │ C   │
│ 1     │ 10492 │ C   │ T   │
│ 1     │ 13273 │ G   │ C   │
│ 1     │ 14462 │ A   │ G   │
│ 1     │ 15484 │ G   │ A   │
│ 1     │ 17005 │ A   │ G   │
│ 1     │ 17020 │ G   │ A   │
│ 1     │ 17257 │ G   │ A   │
│ 1     │ 46633 │ T   │ A   │
│ 1     │ 47267 │ T   │ G   │
│ 1     │ 49157 │ G   │ A   │
│ 1     │ 49298 │ T   │ C   │
│ 1     │ 49554 │ A   │ G   │
│ 1     │ 51479 │ T   │ A   │
│ 1     │ 51979 │ G   │ T   │
│ 1     │ 52238 │ T   │ G   │
│ 1     │ 54050 │ A   │ G   │
│ 1     │ 54353 │ C   │ A   │
│ 1     │ 54490 │ G   │ A   │
│ 1     │ 54591 │ A   │ G   │
└───────┴────

/tmp/ipykernel_2679/74636231.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display_df = display_df.astype(str).applymap(lambda text: str(text).replace('\n', ' ').replace('\r', ' '))


## 5. DOWNLOAD dbNSFP


In [ ]:
# Genos Academic Distribution Server
print('\n' + '=' * 60)
print(f'DOWNLOAD dbNSFP v{DBNSFP_VERSION} - Academic Version (GRCh38)')
print('=' * 60)
if IN_COLAB:
    print('File ~50 GB. Akan disimpan ke Google Drive.')

dbnsfp_file = DBNSFP_DIR / 'dbNSFP5.3a_grch38.gz'
dbnsfp_index = DBNSFP_DIR / 'dbNSFP5.3a_grch38.gz.tbi'
download_file(
    url=f'{DBNSFP_BASE_URL}/dbNSFP5.3a_grch38.gz',
    output_path=dbnsfp_file,
    description=f'dbNSFP v{DBNSFP_VERSION} GRCh38 (GZ)',
)
download_file(
    url=f'{DBNSFP_BASE_URL}/dbNSFP5.3a_grch38.gz.tbi',
    output_path=dbnsfp_index,
    description=f'dbNSFP v{DBNSFP_VERSION} index (TBI)',
)
dbnsfp_preview = preview_table(dbnsfp_file, 'dbNSFP', nrows=20)
save_preview_csv(dbnsfp_preview, PREVIEW_DBNSFP_CSV)



DOWNLOAD dbNSFP v5.3a - Academic Version (GRCh38)
File ~50 GB. Akan disimpan ke Google Drive.

dbNSFP5.3a_grch38.gz sudah ada, melewati download.
   Lokasi : /content/drive/MyDrive/skripsi/data/dbNSFP/dbNSFP5.3a_grch38.gz
   Ukuran : 51.90 GB

dbNSFP5.3a_grch38.gz.tbi sudah ada, melewati download.
   Lokasi : /content/drive/MyDrive/skripsi/data/dbNSFP/dbNSFP5.3a_grch38.gz.tbi
   Ukuran : 881.5 KB

Preview tabel dbNSFP (20 baris teratas):
   Baris (sample): 20 dari total | Kolom ditampilkan: 8 dari 500 total
┌──────┬──────────────┬─────┬─────┬───────┬───────┬──────────────┬──────────┐
│ #chr │ pos(1-based) │ ref │ alt │ aaref │ aaalt │ rs_dbSNP     │ hg19_chr │
├──────┼──────────────┼─────┼─────┼───────┼───────┼──────────────┼──────────┤
│ 1    │ 65565        │ A   │ C   │ M     │ L     │ .            │ 1        │
│ 1    │ 65565        │ A   │ G   │ M     │ V     │ .            │ 1        │
│ 1    │ 65565        │ A   │ T   │ M     │ L     │ .            │ 1        │
│ 1    │ 65566    

/tmp/ipykernel_2679/74636231.py:153: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  display_df = display_df.astype(str).applymap(lambda text: str(text).replace('\n', ' ').replace('\r', ' '))


   Preview CSV disimpan di: /content/drive/MyDrive/skripsi/data/Lampiran_Raw_dbNSFP.csv


## 6. REKAP FILE


In [ ]:
print('\n' + '=' * 65)
print('VERIFIKASI FILE - PIPELINE SKRIPSI')
print('=' * 65)
expected_files = {
    'ClinVar Variant Summary': clinvar_file,
    'WGS West Java VCF': wgs_file,
    'dbNSFP GRCh38 GZ': dbnsfp_file,
    'dbNSFP GRCh38 TBI': dbnsfp_index,
    'Preview ClinVar CSV': PREVIEW_CLINVAR_CSV,
    'Preview WGS CSV': PREVIEW_WGS_CSV,
    'Preview dbNSFP CSV': PREVIEW_DBNSFP_CSV,
}

download_status_rows = []
total_size = 0
all_ready = True
for name, path in expected_files.items():
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    if exists:
        total_size += size
        print(f'  OK  {name}')
        print(f'      {path}')
        print(f'      Ukuran: {human_size(size)}')
    else:
        print(f'  NO  {name}')
        print(f'      {path}')
        all_ready = False
    print()
    download_status_rows.append({
        'name': name,
        'path': str(path),
        'exists': exists,
        'size_bytes': size,
        'size_human': human_size(size) if exists else '0 B',
    })

pd.DataFrame(download_status_rows).to_csv(DOWNLOAD_STATUS_CSV, index=False)
print(f'Download status CSV disimpan di: {DOWNLOAD_STATUS_CSV}')
print('-' * 65)
print(f'Total ukuran: {human_size(total_size)}')
if all_ready:
    print('Semua file siap! Lanjut ke preprocessing.')
else:
    print('Ada file yang belum selesai didownload. Jalankan ulang script ini.')



VERIFIKASI FILE - PIPELINE SKRIPSI
  OK  ClinVar Variant Summary
      /content/drive/MyDrive/skripsi/data/ClinVar/variant_summary.txt.gz
      Ukuran: 441.8 MB

  OK  WGS West Java VCF
      /content/drive/MyDrive/skripsi/data/WGS_WestJava/WGS_WestJava_227.vcf.gz
      Ukuran: 1.71 GB

  OK  dbNSFP GRCh38 GZ
      /content/drive/MyDrive/skripsi/data/dbNSFP/dbNSFP5.3a_grch38.gz
      Ukuran: 51.90 GB

  OK  dbNSFP GRCh38 TBI
      /content/drive/MyDrive/skripsi/data/dbNSFP/dbNSFP5.3a_grch38.gz.tbi
      Ukuran: 881.5 KB

  OK  Preview ClinVar CSV
      /content/drive/MyDrive/skripsi/data/Lampiran_Raw_ClinVar.csv
      Ukuran: 19.1 KB

  OK  Preview WGS CSV
      /content/drive/MyDrive/skripsi/data/Lampiran_Raw_WGS.csv
      Ukuran: 258 B

  OK  Preview dbNSFP CSV
      /content/drive/MyDrive/skripsi/data/Lampiran_Raw_dbNSFP.csv
      Ukuran: 467 B

Download status CSV disimpan di: /content/drive/MyDrive/skripsi/data/download_status.csv
-------------------------------------------------